## EUR + AFR meta-analysis result

## Prepare a UK Biobank panel of 1,000 individuals

In [ ]:
src_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed/result"
out_base="/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_1k"
out_dir="${out_base}/result"
mkdir -p ${out_base}/{code,log} ${out_dir}
cd ${out_base}/code

seed=42     # change for a different random draw; keep fixed for reproducibility

# ---- 1. draw ONE shared set of 1000 individuals (FID IID) from any fileset ----
ref_prefix=$(ls ${src_dir}/*.bed | head -1 | sed 's/\.bed$//')
awk -v s=${seed} 'BEGIN{srand(s)} {print rand()"\t"$1"\t"$2}' "${ref_prefix}.fam" \
    | sort -k1,1g | head -1000 | cut -f2,3 > ${out_base}/keep_1k.txt
echo "keep list: $(wc -l < ${out_base}/keep_1k.txt) individuals"

# ---- 2. list all filesets (handles gaps in the index) ----
ls ${src_dir}/*.bed | sed 's/\.bed$//' > ${out_base}/prefixes.txt
N=$(wc -l < ${out_base}/prefixes.txt)
echo "found ${N} filesets"

cat << 'EOF' > base_trim
#!/bin/bash
#BSUB -J "ukbb_trim[1-NJOBS]"
#BSUB -P acc_ad-omics
#BSUB -q express
#BSUB -n 1
#BSUB -W 1:00
#BSUB -R rusage[mem=16000]
#BSUB -o OUT_BASE/log/ukbb_trim.%J_%I.out
#BSUB -e OUT_BASE/log/ukbb_trim.%J_%I.err

module load plink/1.90

# ---- make sure plink 1.9 is on PATH (edit to your setup) ----

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

out_dir="OUT_DIR"
keep_file="OUT_BASE/keep_1k.txt"
prefixes="OUT_BASE/prefixes.txt"

in_pref=$(sed -n "${LSB_JOBINDEX}p" "${prefixes}")
base=$(basename "${in_pref}")
out_pref="${out_dir}/${base}"

[ -f "${in_pref}.bed" ] || { echo "Missing ${in_pref}.bed"; exit 1; }
[ -f "${keep_file}" ]   || { echo "Missing ${keep_file}"; exit 1; }

echo "Trimming ${base}"
plink \
    --bfile "${in_pref}" \
    --keep "${keep_file}" \
    --keep-allele-order \
    --make-bed \
    --out "${out_pref}"

n=$(wc -l < "${out_pref}.fam")
echo "${base}: ${n} samples" > "${out_pref}.summary.txt"
echo "Done ${base} (${n} samples)"
EOF

sed -e "s|NJOBS|${N}|g" -e "s|OUT_BASE|${out_base}|g" -e "s|OUT_DIR|${out_dir}|g" \
    base_trim > run_ukbb_trim.sh
bsub < run_ukbb_trim.sh

## Sample 1,000 individuals

In [1]:
[EUR_AFR]
parameter: genofile = paths
# One .bim file per region; each region is processed independently.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "72G"
parameter: numThreads = 3

# ---- Simulation knobs --------------------------------------------------------
parameter: n_causal = 1        # number of causal variants per region
parameter: h2g = 0.05          # per-region heritability
parameter: total_h2g = True    # interpret h2g as total (vs per-causal)

# ---- Sample-size sweep (replaces the old weight sweep) -----------------------
# afr_prop is the AFR SAMPLE-SIZE proportion. Cohorts are subsampled so that
#   n_afr = round(afr_prop * total_n),  n_eur = total_n - n_afr.
# Meta-analyzing at the natural (sample-size) weight then makes the mean AND the
# covariance of z_meta follow the SAME mixture R(w) = afr_prop*R_AFR + (1-afr_prop)*R_EUR,
# so R(w) is the exact generative LD at every proportion.
parameter: total_n  = 800      # total sample size (EUR + AFR); must be <= EUR panel size
parameter: afr_prop = 0.66    # AFR sample-size proportion

parameter: container = ""

input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_h2g_{h2g}_prop_{afr_prop}_mix_pop.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    # ---- Libraries ----
    library(tidyverse)      # dplyr, tidyr, purrr, stringr, ggplot2, ...
    library(MASS)
    library(plink2R)
    library(pgenlibr)
    library(vroom)
    library(mixsqp)
    library(matrixStats)
    library(crayon)
    library(tools)
    library(Rfast)
    
    # Genomics / summary-stat tooling
    library(GenomicRanges)
    library(SummarizedExperiment)
    library(Rsamtools)
    library(pecotmr)
    library(susieR)
    library(MungeSumstats)
    
    # Resolve namespace conflicts (load LAST)
    rename <- dplyr::rename
    select <- dplyr::select

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")


    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # ---- Helpers ----
    calculate_sumstat <- function(X, Y) {
      if (is.null(X) || nrow(X) == 0) {
        return(tibble(
          SNP  = character(),
          Beta = numeric(),
          se   = numeric(),
          Freq = numeric(),
          p    = numeric(),
          z    = numeric(),
          N    = integer(),
          chrom = integer(),
          pos   = integer(),
          A2    = character(),
          A1    = character(),
          original_z = numeric(),
          variants_id_original = character(),
          variant_id = character()
        ))
      }
    
      rr <- susieR::univariate_regression(X, Y)
      z  <- rr$betahat / rr$sebetahat
    
      tibble(
        SNP  = colnames(X),
        Beta = rr$betahat,
        se   = rr$sebetahat,
        Freq = colSums(X) / (2 * nrow(X)),
        p    = 2 * pnorm(abs(z), lower.tail = FALSE),
        z    = z,
        N    = nrow(X)
      ) %>%
        separate(SNP, into = c("chrom", "pos", "A2", "A1"), sep = ":", remove = FALSE) %>%
        mutate(
          chrom = as.numeric(str_remove(chrom, "^chr")),
          pos   = as.integer(pos),
          original_z = z,
          variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
          variant_id = variants_id_original
        )
    }
    
    simulate_cohort <- function(G_full, n, B_mat, h2g) {
      if (n == 0) {
        return(list(
          Xmat = G_full[0, , drop = FALSE],
          pheno = NULL,
          target_tb = calculate_sumstat(G_full[0, , drop = FALSE], numeric(0))
        ))
      }
    
      idx <- sample.int(nrow(G_full), n)
      Xmat <- G_full[idx, , drop = FALSE]
      pheno <- sim_multi_traits(G = Xmat, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P
      target_tb <- calculate_sumstat(Xmat, as.numeric(pheno[, 1]))
    
      list(Xmat = Xmat, pheno = pheno, target_tb = target_tb)
    }
    
    # ---- Input ----
    file <- ${_input:r}
    i <- file %>% str_remove("\\.bim$") %>% str_remove(".*/EUR_KG_")
    
    kg_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_1k/result//UKBB_", i)
    kg_data <- plink_process(kg_path)
    
    target_tb <- kg_data$bim %>%
      mutate(
        variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
        variant_id = variants_id_original
      )
    
    chrom <- unique(target_tb$chrom)
    start <- min(target_tb$pos)
    end   <- max(target_tb$pos)
    
    ld_meta_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_hg38/ld_meta_file.tsv"
    region <- sprintf("chr%s:%d-%d", chrom, start, end)
    res_sketch <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
    
    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
    
      idx <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
    
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
    
      colnames(sketch2) <- qced$variants_id_original
    
      list(
        sumstat = qced,
        sketch = sketch2,
        flipped_prop = length(flip_index) / ncol(sketch2)
      )
    }
    
    harm <- sketch_harmonize(
      target_data  = target_tb,
      ref_variants = res_sketch$ref_panel %>% select(-variant_id),
      sketch       = res_sketch$LD_matrix
    )
    
    # ---- Full-panel genotypes ----
    eur_geno_full <- kg_data$bed[, colnames(harm$sketch), drop = FALSE]
    afr_geno_full <- harm$sketch
    p <- ncol(eur_geno_full)
    
    # ---- Parameters ----
    ntrait   <- 1
    ncausal  <- ${n_causal}
    h2g      <- ${h2g}
    total_n  <- ${total_n}
    afr_prop <- ${afr_prop}
    
    # ---- Target cohort sizes ----
    n_afr <- round(afr_prop * total_n)
    n_eur <- total_n - n_afr
    
    if (n_eur > nrow(eur_geno_full)) {
      stop(sprintf("n_eur=%d exceeds EUR panel (%d); lower total_n or afr_prop.",
                   n_eur, nrow(eur_geno_full)))
    }
    if (n_afr > nrow(afr_geno_full)) {
      stop(sprintf("n_afr=%d exceeds AFR panel (%d); lower total_n or raise afr_prop.",
                   n_afr, nrow(afr_geno_full)))
    }
    if (n_eur < 0 || n_afr < 0) {
      stop("Negative cohort size computed; check total_n and afr_prop.")
    }
    
    # ---- Shared causal architecture ----
    set.seed(as.integer(i))
    causal_index <- sample_causal_index(Xmat = eur_geno_full, ncausal = ncausal)
    
    B_mat <- matrix(0, nrow = p, ncol = ntrait)
    B_mat[, 1] <- sim_beta_fix_variant(
      G = eur_geno_full,
      causal_index = causal_index,
      is_h2g_total = FALSE
    )
    
    # ---- Simulate cohorts conditionally ----
    eur_out <- simulate_cohort(eur_geno_full, n_eur, B_mat, h2g)
    afr_out <- simulate_cohort(afr_geno_full, n_afr, B_mat, h2g)
    
    eur_target_tb <- eur_out$target_tb
    afr_target_tb <- afr_out$target_tb
    
    # ---- Panel LD cache ----
    panel_ld_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Mix_simulation/panel_LD"
    dir.create(panel_ld_dir, recursive = TRUE, showWarnings = FALSE)
    panel_ld_path <- file.path(panel_ld_dir, paste0("EUR_KG_", i, ".panelLD.rds"))
    
    if (!file.exists(panel_ld_path)) {
      saveRDS(list(
        EUR_LD_full  = get_cormat(eur_geno_full),
        AFR_LD_full  = get_cormat(afr_geno_full),
        variant_id   = colnames(eur_geno_full),
        n_eur_panel  = nrow(eur_geno_full),
        n_afr_panel  = nrow(afr_geno_full)
      ), panel_ld_path)
    }
    
    # ---- Output bundle ----
    dats <- list(
      eur_target_tb = eur_target_tb,
      afr_target_tb = afr_target_tb,
      eur_Xmat_full = eur_geno_full,
      afr_Xmat_full = afr_geno_full,
      panel_ld_path = panel_ld_path,
      true_vars = causal_index,
      n_eur = n_eur,
      n_afr = n_afr,
      total_n = total_n,
      afr_prop = afr_prop,
      input_path = kg_path
    )
    
    index <- ${_index}
    ss_dir <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Mix_simulation/sumstat/h2g_", h2g,
                     "/causal_", ncausal, "/prop_", afr_prop)
    dir.create(ss_dir, recursive = TRUE, showWarnings = FALSE)
    
    write_tsv(eur_target_tb, file.path(ss_dir, paste0("EUR_", index, ".sumstat.tsv")))
    write_tsv(afr_target_tb, file.path(ss_dir, paste0("AFR_", index, ".sumstat.tsv")))
    
    data_out_path <- ${_output:r}
    dir.create(dirname(data_out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(dats, data_out_path)


ERROR: Error in parse(text = input): <text>:1:1: unexpected '['
1: [
    ^


In [ ]:
#!/bin/bash
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="NEW_EUR_AFR_Mix_simulation"
job_dir="${base_dir}/${job_name}"
total_n=800

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/13.Fig4C_4D_S9B_S9D.AFR_EUR_Meta.ipynb" EUR_AFR \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/kg_trimed_random/*.bim) \
    --n_causal CAUSAL --h2g H2G --afr_prop PROP --total_n TOTAL_N --mem 60G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/prop_PROP/ -j 4
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== Generate + submit one job per (causal, h2g, afr_prop) =====
for causal in 1 2 3; do
  for h2g in 0.025 0.05 0.10 0.15; do
      for prop in 0.0 0.20 0.50 0.80 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|TOTAL_N|${total_n}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
       sb "${output_script}"   # use sb if you aliased
    done
  done
done

## Fine-mapping

In [ ]:
[finemap_sweep]
# Fine-map the EUR+AFR natural-weight meta z at one (h2g, causal, prop).
# prop = AFR sample-size proportion; the realized weight is w = n_afr / total_n.
#   z_meta = (sqrt(n_afr) z_afr + sqrt(n_eur) z_eur) / sqrt(N)     [natural weight]
#   n_mix  = total_n                                              [n_eff(natural) = N]
# LD conditions:
#   *_insample : subsample LD (the exact simulating individuals) -> GOLD generative
#   *_refpanel : full-panel LD (from panel_ld_path)              -> realistic reference-panel
parameter: result_root = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Mix_simulation/result")
parameter: cwd         = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_fine/result")

# sweep parameters
parameter: h2g    = "0.05"
parameter: causal = "1"
parameter: prop   = "0.66"

parameter: job_size = 30
parameter: walltime = "40h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""

# one job per LD sample file in this (h2g, causal, prop) folder
input: [str(x) for x in path(f"{result_root}/h2g_{h2g}/causal_{causal}/prop_{prop}").glob("sample_*_mix_pop.rds")], group_by = 1

output: f'{cwd:a}/h2g_{h2g}/causal_{causal}/prop_{prop}/{_input[0]:bn}.rds'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
    
    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    
    # -----------------------------------------------------------------------------
    # Two susieR builds, each in its own environment so they never clobber.
    # Each susie_*() carries its enclosing env, so internal helper lookups resolve
    # against the build it came from.
    #   susie_hc : Haochen multi-panel / R_finite build -> vanilla + _pop (eb)
    #   susie_xw : Xuewei ld_mismatch build             -> _mix  (eb_mix)
    # -----------------------------------------------------------------------------



    h2g    <- "${h2g}"
    causal <- "${causal}"
    prop   <- "${prop}"

    ld_path  <- ${_input:r}
    out_path <- ${_output:r}
    idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

    cross_cs_maxcorr <- function(fit, R) {
      if (is.null(fit)) return(NA_real_)
      cs <- fit$sets$cs
      if (is.null(cs) || length(cs) < 2) return(NA_real_)
      tops <- vapply(cs, function(m) m[which.max(fit$pip[m])], integer(1))
      cc <- abs(R[tops, tops]); diag(cc) <- NA
      max(cc, na.rm = TRUE)
    }

    save_dropped <- function(reason) {
      saveRDS(list(status = "dropped", reason = reason, prop = prop,
                   index = idx, h2g = h2g, causal = causal, input_path = ld_path),
              out_path)
      cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason))
      quit(save = "no", status = 0)
    }

    ## ---- load sim output ----
    dats  <- readRDS(ld_path)
    n_eur <- dats$n_eur; n_afr <- dats$n_afr; N <- dats$total_n
    w     <- n_afr / N                        # realized natural weight (AFR proportion)

    ## ---- subsample LD (in-sample / generative gold standard) ----
    EUR_sub <- get_cormat(dats$eur_Xmat)
    AFR_sub <- get_cormat(dats$afr_Xmat)
    if (!setequal(colnames(EUR_sub), colnames(AFR_sub)))
      save_dropped("EUR/AFR subsample variant sets differ")
    ref_order <- colnames(EUR_sub)
    AFR_sub   <- AFR_sub[ref_order, ref_order]

    ## ---- full-panel LD (realistic reference-panel case) ----
    panel    <- readRDS(dats$panel_ld_path)
    EUR_full <- panel$EUR_LD_full[ref_order, ref_order]
    AFR_full <- panel$AFR_LD_full[ref_order, ref_order]

    ## ---- natural-weight meta z (direct; both cohorts already aligned) ----
    e_idx <- match(ref_order, dats$eur_target_tb$variant_id)
    a_idx <- match(ref_order, dats$afr_target_tb$variant_id)
     if(nrow(dats$eur_target_tb) == 0){
         z_meta = dats$afr_target_tb$z
     }else if(nrow(dats$afr_target_tb) == 0){
         z_meta = dats$eur_target_tb$z
     }else{
        if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
        z_eur  <- dats$eur_target_tb$z[e_idx]
        z_afr  <- dats$afr_target_tb$z[a_idx]
        z_meta <- (sqrt(n_afr) * z_afr + sqrt(n_eur) * z_eur) / sqrt(N)
    }
    n_mix  <- N                                # n_eff at the natural weight = total_n


    ## ---- generative LD both ways ----
    R_gen_sub  <- w * AFR_sub  + (1 - w) * EUR_sub     # GOLD STANDARD
    R_gen_full <- w * AFR_full + (1 - w) * EUR_full

    fit_or_null <- function(R, tag) tryCatch(
      susie_rss(z = z_meta, R = R, n = n_mix),
      error = function(e) {
        cat(sprintf("[FAIL-FIT] idx %d prop %s %s: %s\n", idx, prop, tag, conditionMessage(e)))
        NULL })

    ## ---- fits ----
    gen_insample <- fit_or_null(R_gen_sub,  "gen_insample")   # GOLD: generative on subsample LD
    gen_refpanel <- fit_or_null(R_gen_full, "gen_refpanel")   # generative on full-panel LD
    eur_refpanel <- fit_or_null(EUR_full,   "eur_refpanel")   # single EUR panel
    afr_refpanel <- fit_or_null(AFR_full,   "afr_refpanel")   # single AFR panel
    mix_refpanel <- fit_or_null(list(AFR_full, EUR_full), "mix_refpanel")  # omega[1]=AFR, [2]=EUR
    om <- if (is.null(mix_refpanel)) NULL else mix_refpanel$omega_weights

    R_manual <- om[1] * AFR_full + om[2] * EUR_full
    manual_refpanel <- fit_or_null(R_manual, "manual_refpanel")
    
    result_list <- list(
      gen_insample = gen_insample,   # <- gold-standard benchmark
      gen_refpanel = gen_refpanel,
      eur_refpanel = eur_refpanel,
      afr_refpanel = afr_refpanel,
      mix_refpanel = mix_refpanel,
      manual_refpanel = manual_refpanel
    )
    
    # per-fit cross-CS correlation, each against the LD it used
    R_used <- list(gen_insample = R_gen_sub, gen_refpanel = R_gen_full,
                       eur_refpanel = EUR_full, afr_refpanel = AFR_full,
                       mix_refpanel = R_gen_full,
                       manual_refpanel = if (!is.null(om) && length(om) == 2)
                                           om[1] * AFR_full + om[2] * EUR_full else R_gen_full)
        max_cs_corr <- imap_dbl(result_list, function(fit, nm) cross_cs_maxcorr(fit, R_used[[nm]]))


    output <- list(
      status        = "ok",
      result_list   = result_list,
      n_sample      = n_mix, n_eur = n_eur, n_afr = n_afr, total_n = N,
      prop = prop, w = w,
      input_path    = ld_path,
      true_vars     = dats$true_vars,
      max_cs_corr   = max_cs_corr,
      omega_weights = om,
      index = idx, h2g = h2g, causal = causal
    )
    dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output, out_path)

    ncs <- function(f) if (is.null(f)) NA else length(f$sets$cs)
    cat(sprintf("[OK] idx %d prop %s  gen(in=%s ref=%s) single(eur=%s afr=%s) mix=%s w=%.3f n=%.0f\n",
                idx, prop, ncs(gen_insample), ncs(gen_refpanel),
                ncs(eur_refpanel), ncs(afr_refpanel), ncs(mix_refpanel), w, n_mix))


In [ ]:
#!/bin/bash
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="NEW_EUR_AFR_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/13.Fig4C_4D_S9B_S9D.AFR_EUR_Meta.ipynb" finemap_sweep \
    --h2g H2G --causal CAUSAL --prop PROP \
    --mem 30G
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== Generate + submit one job per (causal, h2g, prop) =====
# prop = AFR sample-size proportion (must match the prop_* folders the sim wrote).
for causal in 1 2 3; do
  for h2g in 0.025 0.05 0.10 0.15; do
      for prop in 0.0 0.20 0.50 0.80 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
      sb "${output_script}"   # use sb if you aliased
    done
  done
done

## SuSiEx

In [ ]:
[finemap_susiex]
parameter: result_root = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Mix_simulation/result")
parameter: cwd         = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_susiex/result")
parameter: susiex_bin  = "/sc/arion/projects/ad-omics/haochen/rss_qc/susie_x/SuSiEx/bin/SuSiEx"
# sweep parameters
parameter: h2g    = "0.05"
parameter: causal = "1"
parameter: prop   = "0.66"
# SuSiEx knobs
parameter: n_sig       = 10
parameter: level       = 0.95
parameter: min_purity  = 0.5
parameter: pval_thresh = 1e-5
parameter: job_size = 30
parameter: walltime = "40h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""

input: [str(x) for x in path(f"{result_root}/h2g_{h2g}/causal_{causal}/prop_{prop}").glob("sample_*_mix_pop.rds")], group_by = 1
output: f'{cwd:a}/h2g_{h2g}/causal_{causal}/prop_{prop}/{_input[0]:bn}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    suppressMessages({
      library(tidyverse)
      library(Rfast)
    })
    # Project utilities (for get_cormat etc.)
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    select <- dplyr::select
    # safety net if get_cormat is not exported by the sourced utilities
    if (!exists("get_cormat"))
      get_cormat <- function(X) { X <- as.matrix(X); R <- cor(X); dimnames(R) <- list(colnames(X), colnames(X)); R }

    SUSIEX_BIN <- "${susiex_bin}"

    # =====================================================================
    # run_susiex(): write precomputed-LD inputs, call the binary, parse output
    #   pops : named list (e.g. EUR/AFR); each = list(z, R, n)  [z & R in ref_order]
    #   snp  : variant IDs shared across pops (= ref_order)
    #   returns susieR-style: list(sets = list(cs = <list of INDEX vectors>),
    #                              pip = <vector over snp>, n_cs, summary, log)
    # =====================================================================
    run_susiex <- function(pops, snp, bin, workdir, out_name = "sx",
                           n_sig = 10, level = 0.95, min_purity = 0.5,
                           pval_thresh = 1e-5, mult_step = TRUE, threads = 1) {
      m   <- length(snp)
      bp  <- seq_len(m)                       # synthetic, unique, shared across pops
      a1  <- rep("A", m); a2 <- rep("G", m)   # dummy consistent alleles (inputs pre-harmonized)
      dir.create(workdir, showWarnings = FALSE, recursive = TRUE)
      prefixes <- character(length(pops)); ssts <- character(length(pops)); ns <- numeric(length(pops))
      for (i in seq_along(pops)) {
        nm <- names(pops)[i]; p <- pops[[i]]
        z  <- as.numeric(p$z); R <- as.matrix(p$R); diag(R) <- 1
        pref <- file.path(workdir, nm); prefixes[i] <- pref; ns[i] <- p$n
        sst <- data.frame(chr = "1", snp = snp, bp = bp, A1 = a1, A2 = a2,
                          BETA = z, SE = 1, P = 2 * pnorm(-abs(z)), stringsAsFactors = FALSE)
        ssts[i] <- paste0(pref, ".sumstats.txt")
        write.table(sst, ssts[i], sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)
        write.table(data.frame("1", snp, 0, bp, a2, a1), paste0(pref, "_ref.bim"),
                    sep = "\t", quote = FALSE, row.names = FALSE, col.names = FALSE)   # col5=a2,col6=a1
        write.table(data.frame(CHR = "1", SNP = snp, A1 = a2, A2 = a1, MAF = 0.3,
                               NCHROBS = round(2 * p$n)), paste0(pref, "_frq.frq"),
                    sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE)     # col3=a2,col4=a1
        con <- file(paste0(pref, ".ld.bin"), "wb"); writeBin(as.numeric(as.vector(R)), con, size = 4); close(con)
      }
      cc <- function(v) paste(rep(v, length(pops)), collapse = ",")
      args <- c(
        paste0("--sst_file=", paste(ssts, collapse = ",")),
        paste0("--n_gwas=",   paste(ns, collapse = ",")),
        paste0("--ld_file=",  paste(prefixes, collapse = ",")),
        paste0("--out_dir=",  workdir), paste0("--out_name=", out_name),
        "--chr=1", paste0("--bp=", min(bp), ",", max(bp)),
        paste0("--chr_col=", cc(1)), paste0("--snp_col=", cc(2)), paste0("--bp_col=", cc(3)),
        paste0("--a1_col=", cc(4)), paste0("--a2_col=", cc(5)), paste0("--eff_col=", cc(6)),
        paste0("--se_col=", cc(7)), paste0("--pval_col=", cc(8)),
        paste0("--n_sig=", n_sig), paste0("--level=", level),
        paste0("--min_purity=", min_purity), paste0("--pval_thresh=", pval_thresh),
        "--maf=0", paste0("--mult-step=", ifelse(mult_step, "True", "False")),
        "--keep-ambig=True", "--plink=plink", paste0("--threads=", threads))
      log <- suppressWarnings(system2(bin, args, stdout = TRUE, stderr = TRUE))

      # ---- parse .cs -> credible sets as INDICES into `snp` (=ref_order) ----
      cs_path <- file.path(workdir, paste0(out_name, ".cs"))
      sets <- list()
      if (file.exists(cs_path)) {
        raw <- readLines(cs_path); raw <- raw[!grepl("^#", raw)]; raw <- raw[nzchar(trimws(raw))]
        if (length(raw) && !grepl("^(NULL|FAIL)", raw[1])) {
          cs <- read.table(text = paste(raw, collapse = "\n"), header = TRUE, sep = "\t",
                           check.names = FALSE, stringsAsFactors = FALSE)
          sets <- lapply(split(cs$SNP, cs$CS_ID), function(ids) match(as.character(ids), snp))
          names(sets) <- paste0("L", names(sets))
        }
      }
      # ---- parse .snp -> overall PIP over `snp` ----
      pip <- setNames(numeric(m), snp)
      snp_path <- file.path(workdir, paste0(out_name, ".snp"))
      if (file.exists(snp_path)) {
        sp <- try(read.table(snp_path, header = TRUE, sep = "\t", check.names = FALSE,
                             stringsAsFactors = FALSE), silent = TRUE)
        if (!inherits(sp, "try-error") && "SNP" %in% names(sp)) {
          pcols <- grep("^PIP\\(CS", names(sp), value = TRUE)
          if (length(pcols)) {
            ov <- 1 - apply(1 - as.matrix(sp[, pcols, drop = FALSE]), 1, prod)
            pip[match(as.character(sp$SNP), snp)] <- ov
          }
        }
      }
      list(sets = list(cs = sets), pip = unname(pip), n_cs = length(sets),
           summary_file = file.path(workdir, paste0(out_name, ".summary")), log = log)
    }
    ncs <- function(f) if (is.null(f)) NA_integer_ else length(f$sets$cs)

    # =====================================================================
    # per-input driver (mirrors finemap_sweep bookkeeping)
    # =====================================================================
    h2g <- "${h2g}"; causal <- "${causal}"; prop <- "${prop}"
    ld_path  <- ${_input:r}
    out_path <- ${_output:r}
    idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

    save_dropped <- function(reason) {
      saveRDS(list(status = "dropped", reason = reason, prop = prop, index = idx,
                   h2g = h2g, causal = causal, input_path = ld_path), out_path)
      cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason)); quit(save = "no", status = 0)
    }

    dats  <- readRDS(ld_path)
    n_eur <- dats$n_eur; n_afr <- dats$n_afr; N <- dats$total_n; w <- n_afr / N
    eur_ok <- nrow(dats$eur_target_tb) > 0
    afr_ok <- nrow(dats$afr_target_tb) > 0
    if (!eur_ok && !afr_ok) save_dropped("both target tables empty")

    ## ---- in-sample (subsample) LD, define ref_order ----
    subLD <- list()
    if (eur_ok) subLD$EUR <- get_cormat(dats$eur_Xmat)
    if (afr_ok) subLD$AFR <- get_cormat(dats$afr_Xmat)
    if (eur_ok && afr_ok && !setequal(colnames(subLD$EUR), colnames(subLD$AFR)))
      save_dropped("EUR/AFR subsample variant sets differ")
    ref_order <- colnames(subLD[[1]])
    subLD <- lapply(subLD, function(M) M[ref_order, ref_order])

    ## ---- full reference-panel LD ----
    panel  <- readRDS(dats$panel_ld_path)
    fullLD <- list()
    if (eur_ok) fullLD$EUR <- panel$EUR_LD_full[ref_order, ref_order]
    if (afr_ok) fullLD$AFR <- panel$AFR_LD_full[ref_order, ref_order]

    ## ---- per-population z aligned to ref_order ----
    z <- list(); nn <- list()
    if (eur_ok) { z$EUR <- dats$eur_target_tb$z[match(ref_order, dats$eur_target_tb$variant_id)]; nn$EUR <- n_eur }
    if (afr_ok) { z$AFR <- dats$afr_target_tb$z[match(ref_order, dats$afr_target_tb$variant_id)]; nn$AFR <- n_afr }

    mkpops <- function(LDset) setNames(
      lapply(names(LDset), function(k) list(z = z[[k]], R = LDset[[k]], n = nn[[k]])), names(LDset))

    base_tmp <- file.path(dirname(out_path), paste0(".sxtmp_", tools::file_path_sans_ext(basename(out_path))))
    fit_regime <- function(LDset, tag) {
      wd <- file.path(base_tmp, tag)
      out <- tryCatch(
        run_susiex(mkpops(LDset), snp = ref_order, bin = SUSIEX_BIN, workdir = wd,
                   out_name = tag, n_sig = ${n_sig}, level = ${level},
                   min_purity = ${min_purity}, pval_thresh = ${pval_thresh}),
        error = function(e) { cat(sprintf("[FAIL-FIT] idx %d %s: %s\n", idx, tag, conditionMessage(e))); NULL })
      out
    }

    susiex_insample <- fit_regime(subLD,  "susiex_insample")   # gold-standard LD
    susiex_refpanel <- fit_regime(fullLD, "susiex_refpanel")   # realistic reference-panel LD

    result_list <- list(susiex_insample = susiex_insample,
                        susiex_refpanel = susiex_refpanel)

    output <- list(
      status        = "ok",
      result_list   = result_list,          # each: $sets$cs (indices), $pip (over ref_order)
      ref_order     = ref_order,            # names for the indices
      true_vars     = dats$true_vars,       # indices, comparable to $sets$cs
      n_eur = n_eur, n_afr = n_afr, total_n = N, prop = prop, w = w,
      pops_used     = names(z),
      input_path    = ld_path,
      index = idx, h2g = h2g, causal = causal
    )
    dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output, out_path)
    unlink(base_tmp, recursive = TRUE)      # remove scratch SuSiEx files; comment out to keep

    cat(sprintf("[OK] idx %d prop %s  susiex(insample=%s refpanel=%s)  pops=%s  w=%.3f n=%.0f\n",
                idx, prop, ncs(susiex_insample), ncs(susiex_refpanel),
                paste(names(z), collapse = "+"), w, N))

In [ ]:
#!/bin/bash
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="NEW_EUR_AFR_susiex"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/13.Fig4C_4D_S9B_S9D.AFR_EUR_Meta.ipynb" finemap_susiex \
    --h2g H2G --causal CAUSAL --prop PROP \
    --mem 30G
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== Generate + submit one job per (causal, h2g, prop) =====
# prop = AFR sample-size proportion (must match the prop_* folders the sim wrote).
for causal in 1 2 3; do
  for h2g in 0.0125 0.025 0.0375; do
      for prop in 0.30 0.50 0.70; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
        sb "${output_script}"   # use sb if you aliased
    done
  done
done

## Summary code

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_fine/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J susie_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/log/summary.%J.out
#BSUB -e WORK_DIR/log/summary.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai//14.Fig4.AFR_EUR_Meta_v1_and_Summary_Steps.ipynb" susie_mix_summary \
    --folder /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_fine/result/ \
    --mem 20G \
    --cwd WORK_DIR/summary/


EOF

base_sh="base_script"

output_script="summary_susie.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_susiex/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J susiex_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=20000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/log/summary.%J.out
#BSUB -e WORK_DIR/log/summary.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/14.Fig4.AFR_EUR_Meta_v1_and_Summary_Steps.ipynb" susiex_mix_summary \
    --folder /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_susiex/ \
    --mem 20G \
    --cwd WORK_DIR/summary/


EOF

base_sh="base_script"

output_script="summary_susiex.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}

## Heterogeneous effects

In [ ]:
[EUR_AFR_hetero]
parameter: genofile = paths
# One .bim file per region; each region is processed independently.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "100h"
parameter: mem = "72G"
parameter: numThreads = 3

# ---- Simulation knobs --------------------------------------------------------
parameter: n_causal = 1        # number of causal variants per region
parameter: h2g = 0.05          # per-region heritability
parameter: total_h2g = True    # interpret h2g as total (vs per-causal)

# ---- Sample-size sweep (replaces the old weight sweep) -----------------------
# afr_prop is the AFR SAMPLE-SIZE proportion. Cohorts are subsampled so that
#   n_afr = round(afr_prop * total_n),  n_eur = total_n - n_afr.
# Meta-analyzing at the natural (sample-size) weight then makes the mean AND the
# covariance of z_meta follow the SAME mixture R(w) = afr_prop*R_AFR + (1-afr_prop)*R_EUR,
# so R(w) is the exact generative LD at every proportion.
parameter: total_n  = 800      # total sample size (EUR + AFR); must be <= EUR panel size
parameter: afr_prop = 0.66    # AFR sample-size proportion

parameter: container = ""

input: genofile, group_by = 1
output: f'{cwd:a}/sample_{_index}_ncausal_{n_causal}_h2g_{h2g}_prop_{afr_prop}_mix_pop.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R:  expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    # ---- Libraries ----
    library(tidyverse)      # dplyr, tidyr, purrr, stringr, ggplot2, ...
    library(MASS)
    library(plink2R)
    library(pgenlibr)
    library(vroom)
    library(mixsqp)
    library(matrixStats)
    library(crayon)
    library(tools)
    library(Rfast)
    
    # Genomics / summary-stat tooling
    library(GenomicRanges)
    library(SummarizedExperiment)
    library(Rsamtools)
    library(pecotmr)
    library(susieR)
    library(MungeSumstats)
    
    # Resolve namespace conflicts (load LAST)
    rename <- dplyr::rename
    select <- dplyr::select

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")


    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # ---- Helpers ----
    calculate_sumstat <- function(X, Y) {
      if (is.null(X) || nrow(X) == 0) {
        return(tibble(
          SNP  = character(),
          Beta = numeric(),
          se   = numeric(),
          Freq = numeric(),
          p    = numeric(),
          z    = numeric(),
          N    = integer(),
          chrom = integer(),
          pos   = integer(),
          A2    = character(),
          A1    = character(),
          original_z = numeric(),
          variants_id_original = character(),
          variant_id = character()
        ))
      }
    
      rr <- susieR::univariate_regression(X, Y)
      z  <- rr$betahat / rr$sebetahat
    
      tibble(
        SNP  = colnames(X),
        Beta = rr$betahat,
        se   = rr$sebetahat,
        Freq = colSums(X) / (2 * nrow(X)),
        p    = 2 * pnorm(abs(z), lower.tail = FALSE),
        z    = z,
        N    = nrow(X)
      ) %>%
        separate(SNP, into = c("chrom", "pos", "A2", "A1"), sep = ":", remove = FALSE) %>%
        mutate(
          chrom = as.numeric(str_remove(chrom, "^chr")),
          pos   = as.integer(pos),
          original_z = z,
          variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
          variant_id = variants_id_original
        )
    }
    
    simulate_cohort <- function(G_full, n, B_mat, h2g) {
      if (n == 0) {
        return(list(
          Xmat = G_full[0, , drop = FALSE],
          pheno = NULL,
          target_tb = calculate_sumstat(G_full[0, , drop = FALSE], numeric(0))
        ))
      }
    
      idx <- sample.int(nrow(G_full), n)
      Xmat <- G_full[idx, , drop = FALSE]
      pheno <- sim_multi_traits(G = Xmat, B = B_mat, h2g = h2g, is_h2g_total = FALSE)$P
      target_tb <- calculate_sumstat(Xmat, as.numeric(pheno[, 1]))
    
      list(Xmat = Xmat, pheno = pheno, target_tb = target_tb)
    }
    
    # ---- Input ----
    file <- ${_input:r}
    i <- file %>% str_remove("\\.bim$") %>% str_remove(".*/EUR_KG_")
    
    kg_path <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ukbb_unrelated_trimed_sample_1k/result//UKBB_", i)
    kg_data <- plink_process(kg_path)
    
    target_tb <- kg_data$bim %>%
      mutate(
        variants_id_original = paste0("chr", chrom, ":", pos, ":", A2, ":", A1),
        variant_id = variants_id_original
      )
    
    chrom <- unique(target_tb$chrom)
    start <- min(target_tb$pos)
    end   <- max(target_tb$pos)
    
    ld_meta_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/1000G_AFR_hg38/ld_meta_file.tsv"
    region <- sprintf("chr%s:%d-%d", chrom, start, end)
    res_sketch <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
    
    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
    
      idx <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
    
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
    
      colnames(sketch2) <- qced$variants_id_original
    
      list(
        sumstat = qced,
        sketch = sketch2,
        flipped_prop = length(flip_index) / ncol(sketch2)
      )
    }
    
    harm <- sketch_harmonize(
      target_data  = target_tb,
      ref_variants = res_sketch$ref_panel %>% select(-variant_id),
      sketch       = res_sketch$LD_matrix
    )
    
    # ---- Full-panel genotypes ----
    eur_geno_full <- kg_data$bed[, colnames(harm$sketch), drop = FALSE]
    afr_geno_full <- harm$sketch
    p <- ncol(eur_geno_full)
    
    # ---- Parameters ----
    ntrait   <- 1
    ncausal  <- ${n_causal}
    h2g      <- ${h2g}
    total_n  <- ${total_n}
    afr_prop <- ${afr_prop}
    
    # ---- Target cohort sizes ----
    n_afr <- round(afr_prop * total_n)
    n_eur <- total_n - n_afr
    
    if (n_eur > nrow(eur_geno_full)) {
      stop(sprintf("n_eur=%d exceeds EUR panel (%d); lower total_n or afr_prop.",
                   n_eur, nrow(eur_geno_full)))
    }
    if (n_afr > nrow(afr_geno_full)) {
      stop(sprintf("n_afr=%d exceeds AFR panel (%d); lower total_n or raise afr_prop.",
                   n_afr, nrow(afr_geno_full)))
    }
    if (n_eur < 0 || n_afr < 0) {
      stop("Negative cohort size computed; check total_n and afr_prop.")
    }
    
    # ---- Shared causal architecture ----
    # ---- Independent causal architecture per cohort (NOT shared) ----
    set.seed(as.integer(i))

    causal_eur <- sample_causal_index(Xmat = eur_geno_full, ncausal = ncausal)

    # draw AFR causal disjoint from EUR (redraw until no overlap)
    repeat {
      causal_afr <- sample_causal_index(Xmat = afr_geno_full, ncausal = ncausal)
      if (length(intersect(causal_afr, causal_eur)) == 0) break
    }

    # separate effect vectors, each calibrated to its own cohort's genotypes
    B_eur <- matrix(0, nrow = p, ncol = ntrait)
    B_eur[, 1] <- sim_beta_fix_variant(
      G = eur_geno_full, causal_index = causal_eur, is_h2g_total = FALSE
    )

    B_afr <- matrix(0, nrow = p, ncol = ntrait)
    B_afr[, 1] <- sim_beta_fix_variant(
      G = afr_geno_full, causal_index = causal_afr, is_h2g_total = FALSE
    )

    # ---- Simulate cohorts with their OWN architecture ----
    eur_out <- simulate_cohort(eur_geno_full, n_eur, B_eur, h2g)
    afr_out <- simulate_cohort(afr_geno_full, n_afr, B_afr, h2g)

    eur_target_tb <- eur_out$target_tb
    afr_target_tb <- afr_out$target_tb
    
    # ---- Panel LD cache ----
    panel_ld_dir  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero/panel_LD"
    dir.create(panel_ld_dir, recursive = TRUE, showWarnings = FALSE)
    panel_ld_path <- file.path(panel_ld_dir, paste0("EUR_KG_", i, ".panelLD.rds"))
    
    if (!file.exists(panel_ld_path)) {
      saveRDS(list(
        EUR_LD_full  = get_cormat(eur_geno_full),
        AFR_LD_full  = get_cormat(afr_geno_full),
        variant_id   = colnames(eur_geno_full),
        n_eur_panel  = nrow(eur_geno_full),
        n_afr_panel  = nrow(afr_geno_full)
      ), panel_ld_path)
    }
    
    # ---- Output bundle ----
    dats <- list(
      eur_target_tb = eur_target_tb,
      afr_target_tb = afr_target_tb,
      eur_Xmat_full = eur_geno_full,
      afr_Xmat_full = afr_geno_full,
      panel_ld_path = panel_ld_path,
      true_vars_eur = causal_eur,
      true_vars_afr = causal_afr,
      n_eur = n_eur,
      n_afr = n_afr,
      total_n = total_n,
      afr_prop = afr_prop,
      input_path = kg_path
    )
    
    index <- ${_index}
    ss_dir <- paste0("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero/sumstat/h2g_", h2g,
                     "/causal_", ncausal, "/prop_", afr_prop)
    dir.create(ss_dir, recursive = TRUE, showWarnings = FALSE)
    
    write_tsv(eur_target_tb, file.path(ss_dir, paste0("EUR_", index, ".sumstat.tsv")))
    write_tsv(afr_target_tb, file.path(ss_dir, paste0("AFR_", index, ".sumstat.tsv")))
    
    data_out_path <- ${_output:r}
    dir.create(dirname(data_out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(dats, data_out_path)


In [ ]:
#!/bin/bash
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="NEW_EUR_AFR_Hetero"
job_dir="${base_dir}/${job_name}"
total_n=800

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base SLURM script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/13.Fig4C_4D_S9B_S9D.AFR_EUR_Meta.ipynb" EUR_AFR_hetero \
    --genofile $(echo /sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/kg_trimed_random/*.bim) \
    --n_causal CAUSAL --h2g H2G --afr_prop PROP --total_n TOTAL_N --mem 60G \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL/prop_PROP/ -j 4
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== Generate + submit one job per (causal, h2g, afr_prop) =====
for causal in 1 2 3; do
  for h2g in 0.025 0.0375 0.05; do
      for prop in 0.0 0.30 0.50 0.70 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|TOTAL_N|${total_n}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
       sb "${output_script}"   # use sb if you aliased
    done
  done
done

In [ ]:
[fine_prop]
# Fine-map the EUR+AFR natural-weight meta z at one (h2g, causal, prop).
# prop = AFR sample-size proportion; the realized weight is w = n_afr / total_n.
#   z_meta = (sqrt(n_afr) z_afr + sqrt(n_eur) z_eur) / sqrt(N)     [natural weight]
#   n_mix  = total_n                                              [n_eff(natural) = N]
# LD conditions:
#   *_insample : subsample LD (the exact simulating individuals) -> GOLD generative
#   *_refpanel : full-panel LD (from panel_ld_path)              -> realistic reference-panel
parameter: result_root = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero/result")
parameter: cwd         = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero_Fine/result")

# sweep parameters
parameter: h2g    = "0.05"
parameter: causal = "1"
parameter: prop   = "0.66"

parameter: job_size = 30
parameter: walltime = "40h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""

# one job per LD sample file in this (h2g, causal, prop) folder
input: [str(x) for x in path(f"{result_root}/h2g_{h2g}/causal_{causal}/prop_{prop}").glob("sample_*_mix_pop.rds")], group_by = 1

output: f'{cwd:a}/h2g_{h2g}/causal_{causal}/prop_{prop}/{_input[0]:bn}.rds'

task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    
    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
        select <- dplyr::select
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")

    h2g <- "${h2g}"; causal <- "${causal}"; prop <- "${prop}"
    ld_path  <- ${_input:r}
    out_path <- ${_output:r}
    idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

    save_dropped <- function(reason) {
      saveRDS(list(status = "dropped", reason = reason, prop = prop,
                   index = idx, h2g = h2g, causal = causal, input_path = ld_path), out_path)
      cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason))
      quit(save = "no", status = 0)
    }
    fit_or_null <- function(R, tag) tryCatch(
      susie_rss(z = z_meta, R = R, n = n_mix),
      error = function(e) { cat(sprintf("[FAIL-FIT] idx %d prop %s %s: %s\n",
                                        idx, prop, tag, conditionMessage(e))); NULL })

    ## ---- load sim output ----
    dats  <- readRDS(ld_path)
    n_eur <- dats$n_eur; n_afr <- dats$n_afr; N <- dats$total_n
    w     <- n_afr / N                          # true AFR proportion

    ## ---- full-panel LD (the reference-panel mixture is estimated on these) ----
    panel     <- readRDS(dats$panel_ld_path)
    ref_order <- panel$variant_id
    EUR_full  <- panel$EUR_LD_full[ref_order, ref_order]
    AFR_full  <- panel$AFR_LD_full[ref_order, ref_order]

    ## ---- natural-weight meta z ----
    e_idx <- match(ref_order, dats$eur_target_tb$variant_id)
    a_idx <- match(ref_order, dats$afr_target_tb$variant_id)
    if (nrow(dats$eur_target_tb) == 0) {
      z_meta <- dats$afr_target_tb$z[a_idx]
    } else if (nrow(dats$afr_target_tb) == 0) {
      z_meta <- dats$eur_target_tb$z[e_idx]
    } else {
      if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
      z_meta <- (sqrt(n_afr) * dats$afr_target_tb$z[a_idx] +
                 sqrt(n_eur) * dats$eur_target_tb$z[e_idx]) / sqrt(N)
    }
    n_mix <- N

    ## ---- ONLY the mixture fit, to estimate omega ----
    mix_refpanel <- fit_or_null(list(AFR_full, EUR_full), "mix_refpanel")  # omega[1]=AFR, [2]=EUR
    om <- if (is.null(mix_refpanel)) NULL else mix_refpanel$omega_weights

    output <- list(
      status         = "ok",
      omega_weights  = om,                 # c(AFR, EUR)
      est_afr_prop   = if (is.null(om)) NA_real_ else om[1],
      w              = w,                  # true AFR proportion (for comparison)
      prop = prop, n_eur = n_eur, n_afr = n_afr, total_n = N,
      input_path = ld_path, index = idx, h2g = h2g, causal = causal
    )
    dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
    saveRDS(output, out_path)

    cat(sprintf("[OK] idx %d prop %s  est_afr=%.3f  true_afr=%.3f\n",
                idx, prop, output$est_afr_prop, w))


In [ ]:
#!/bin/bash
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="NEW_EUR_AFR_hetero_fine"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,log}
cd "${job_dir}/code"

# ===== Base SLURM script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 42:00
#BSUB -o /WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME.%J.err
export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/13.Fig4C_4D_S9B_S9D.AFR_EUR_Meta.ipynb" fine_prop \
    --h2g H2G --causal CAUSAL --prop PROP \
    --mem 30G
EOF
chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== Generate + submit one job per (causal, h2g, prop) =====
# prop = AFR sample-size proportion (must match the prop_* folders the sim wrote).
for causal in 1 2 3; do
  for h2g in 0.025 0.0375 0.05; do
      for prop in 0.0 0.30 0.50 0.70 1.0; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}_prop_${prop}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|PROP|${prop}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"
       sb "${output_script}"   # use sb if you aliased
    done
  done
done

In [11]:
library(plink2R)
library(tidyverse)
library(MASS)
library(crayon)
library(pecotmr)
library(susieR)
library(vroom)
library(pgenlibr)
library(Rfast)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")

h2g <- "0.025"; causal <- "2"; prop <- "0.30"
ld_path  <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero/result/h2g_0.025/causal_2/prop_0.30/sample_117_ncausal_2_h2g_0.025_prop_0.3_mix_pop.rds'
out_path <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero_Fine/result/h2g_0.025/causal_2/prop_0.30/sample_117_ncausal_2_h2g_0.025_prop_0.3_mix_pop.rds'
idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

save_dropped <- function(reason) {
  saveRDS(list(status = "dropped", reason = reason, prop = prop,
               index = idx, h2g = h2g, causal = causal, input_path = ld_path), out_path)
  cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason))
  quit(save = "no", status = 0)
}
fit_or_null <- function(R, tag) tryCatch(
  susie_rss(z = z_meta, R = R, n = n_mix),
  error = function(e) { cat(sprintf("[FAIL-FIT] idx %d prop %s %s: %s\n",
                                    idx, prop, tag, conditionMessage(e))); NULL })

## ---- load sim output ----
dats  <- readRDS(ld_path)
n_eur <- dats$n_eur; n_afr <- dats$n_afr; N <- dats$total_n
w     <- n_afr / N                          # true AFR proportion

## ---- full-panel LD (the reference-panel mixture is estimated on these) ----
panel     <- readRDS(dats$panel_ld_path)
ref_order <- panel$variant_id
EUR_full  <- panel$EUR_LD_full[ref_order, ref_order]
AFR_full  <- panel$AFR_LD_full[ref_order, ref_order]

## ---- natural-weight meta z ----
e_idx <- match(ref_order, dats$eur_target_tb$variant_id)
a_idx <- match(ref_order, dats$afr_target_tb$variant_id)
if (nrow(dats$eur_target_tb) == 0) {
  z_meta <- dats$afr_target_tb$z[a_idx]
} else if (nrow(dats$afr_target_tb) == 0) {
  z_meta <- dats$eur_target_tb$z[e_idx]
} else {
  if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
  z_meta <- (sqrt(n_afr) * dats$afr_target_tb$z[a_idx] +
             sqrt(n_eur) * dats$eur_target_tb$z[e_idx]) / sqrt(N)
}
n_mix <- N

## ---- ONLY the mixture fit, to estimate omega ----
mix_refpanel <- fit_or_null(list(AFR_full, EUR_full), "mix_refpanel")  # omega[1]=AFR, [2]=EUR
om <- if (is.null(mix_refpanel)) NULL else mix_refpanel$omega_weights

output <- list(
  status         = "ok",
  omega_weights  = om,                 # c(AFR, EUR)
  est_afr_prop   = if (is.null(om)) NA_real_ else om[1],
  w              = w,                  # true AFR proportion (for comparison)
  prop = prop, n_eur = n_eur, n_afr = n_afr, total_n = N,
  input_path = ld_path, index = idx, h2g = h2g, causal = causal
)
dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
saveRDS(output, out_path)

cat(sprintf("[OK] idx %d prop %s  est_afr=%.3f  true_afr=%.3f\n",
            idx, prop, output$est_afr_prop, w))

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: multi-panel mixture weight updates change R(omega) each iteration, which prevents ELBO monotonicity.



[OK] idx 117 prop 0.30  est_afr=0.250  true_afr=0.300


In [12]:
library(plink2R)
library(tidyverse)
library(MASS)
library(crayon)
library(pecotmr)
library(susieR)
library(vroom)
library(pgenlibr)
library(Rfast)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")

h2g <- "0.025"; causal <- "2"; prop <- "0.30"
ld_path  <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero/result/h2g_0.025/causal_2/prop_0.30/sample_127_ncausal_2_h2g_0.025_prop_0.3_mix_pop.rds'
out_path <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero_Fine/result/h2g_0.025/causal_2/prop_0.30/sample_127_ncausal_2_h2g_0.025_prop_0.3_mix_pop.rds'
idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

save_dropped <- function(reason) {
  saveRDS(list(status = "dropped", reason = reason, prop = prop,
               index = idx, h2g = h2g, causal = causal, input_path = ld_path), out_path)
  cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason))
  quit(save = "no", status = 0)
}
fit_or_null <- function(R, tag) tryCatch(
  susie_rss(z = z_meta, R = R, n = n_mix),
  error = function(e) { cat(sprintf("[FAIL-FIT] idx %d prop %s %s: %s\n",
                                    idx, prop, tag, conditionMessage(e))); NULL })

## ---- load sim output ----
dats  <- readRDS(ld_path)
n_eur <- dats$n_eur; n_afr <- dats$n_afr; N <- dats$total_n
w     <- n_afr / N                          # true AFR proportion

## ---- full-panel LD (the reference-panel mixture is estimated on these) ----
panel     <- readRDS(dats$panel_ld_path)
ref_order <- panel$variant_id
EUR_full  <- panel$EUR_LD_full[ref_order, ref_order]
AFR_full  <- panel$AFR_LD_full[ref_order, ref_order]

## ---- natural-weight meta z ----
e_idx <- match(ref_order, dats$eur_target_tb$variant_id)
a_idx <- match(ref_order, dats$afr_target_tb$variant_id)
if (nrow(dats$eur_target_tb) == 0) {
  z_meta <- dats$afr_target_tb$z[a_idx]
} else if (nrow(dats$afr_target_tb) == 0) {
  z_meta <- dats$eur_target_tb$z[e_idx]
} else {
  if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
  z_meta <- (sqrt(n_afr) * dats$afr_target_tb$z[a_idx] +
             sqrt(n_eur) * dats$eur_target_tb$z[e_idx]) / sqrt(N)
}
n_mix <- N

## ---- ONLY the mixture fit, to estimate omega ----
mix_refpanel <- fit_or_null(list(AFR_full, EUR_full), "mix_refpanel")  # omega[1]=AFR, [2]=EUR
om <- if (is.null(mix_refpanel)) NULL else mix_refpanel$omega_weights

output <- list(
  status         = "ok",
  omega_weights  = om,                 # c(AFR, EUR)
  est_afr_prop   = if (is.null(om)) NA_real_ else om[1],
  w              = w,                  # true AFR proportion (for comparison)
  prop = prop, n_eur = n_eur, n_afr = n_afr, total_n = N,
  input_path = ld_path, index = idx, h2g = h2g, causal = causal
)
dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
saveRDS(output, out_path)

cat(sprintf("[OK] idx %d prop %s  est_afr=%.3f  true_afr=%.3f\n",
            idx, prop, output$est_afr_prop, w))

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: multi-panel mixture weight updates change R(omega) each iteration, which prevents ELBO monotonicity.


Warning message:
“IBSS algorithm did not converge in 50 iterations!”


[OK] idx 127 prop 0.30  est_afr=0.361  true_afr=0.300


In [13]:
library(plink2R)
library(tidyverse)
library(MASS)
library(crayon)
library(pecotmr)
library(susieR)
library(vroom)
library(pgenlibr)
library(Rfast)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")

h2g <- "0.025"; causal <- "2"; prop <- "0.30"
ld_path  <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero/result/h2g_0.025/causal_2/prop_0.70/sample_127_ncausal_2_h2g_0.025_prop_0.7_mix_pop.rds'
out_path <- '/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/NEW_EUR_AFR_Hetero_Fine/result/h2g_0.025/causal_2/prop_0.70/sample_127_ncausal_2_h2g_0.025_prop_0.7_mix_pop.rds'
idx      <- as.integer(str_match(basename(ld_path), "sample_([0-9]+)_")[, 2])

save_dropped <- function(reason) {
  saveRDS(list(status = "dropped", reason = reason, prop = prop,
               index = idx, h2g = h2g, causal = causal, input_path = ld_path), out_path)
  cat(sprintf("[DROP] idx %d prop %s: %s\n", idx, prop, reason))
  quit(save = "no", status = 0)
}
fit_or_null <- function(R, tag) tryCatch(
  susie_rss(z = z_meta, R = R, n = n_mix),
  error = function(e) { cat(sprintf("[FAIL-FIT] idx %d prop %s %s: %s\n",
                                    idx, prop, tag, conditionMessage(e))); NULL })

## ---- load sim output ----
dats  <- readRDS(ld_path)
n_eur <- dats$n_eur; n_afr <- dats$n_afr; N <- dats$total_n
w     <- n_afr / N                          # true AFR proportion

## ---- full-panel LD (the reference-panel mixture is estimated on these) ----
panel     <- readRDS(dats$panel_ld_path)
ref_order <- panel$variant_id
EUR_full  <- panel$EUR_LD_full[ref_order, ref_order]
AFR_full  <- panel$AFR_LD_full[ref_order, ref_order]

## ---- natural-weight meta z ----
e_idx <- match(ref_order, dats$eur_target_tb$variant_id)
a_idx <- match(ref_order, dats$afr_target_tb$variant_id)
if (nrow(dats$eur_target_tb) == 0) {
  z_meta <- dats$afr_target_tb$z[a_idx]
} else if (nrow(dats$afr_target_tb) == 0) {
  z_meta <- dats$eur_target_tb$z[e_idx]
} else {
  if (anyNA(e_idx) || anyNA(a_idx)) save_dropped("sumstat variants do not cover ref_order")
  z_meta <- (sqrt(n_afr) * dats$afr_target_tb$z[a_idx] +
             sqrt(n_eur) * dats$eur_target_tb$z[e_idx]) / sqrt(N)
}
n_mix <- N

## ---- ONLY the mixture fit, to estimate omega ----
mix_refpanel <- fit_or_null(list(AFR_full, EUR_full), "mix_refpanel")  # omega[1]=AFR, [2]=EUR
om <- if (is.null(mix_refpanel)) NULL else mix_refpanel$omega_weights

output <- list(
  status         = "ok",
  omega_weights  = om,                 # c(AFR, EUR)
  est_afr_prop   = if (is.null(om)) NA_real_ else om[1],
  w              = w,                  # true AFR proportion (for comparison)
  prop = prop, n_eur = n_eur, n_afr = n_afr, total_n = N,
  input_path = ld_path, index = idx, h2g = h2g, causal = causal
)
dir.create(dirname(out_path), recursive = TRUE, showWarnings = FALSE)
saveRDS(output, out_path)

cat(sprintf("[OK] idx %d prop %s  est_afr=%.3f  true_afr=%.3f\n",
            idx, prop, output$est_afr_prop, w))

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: multi-panel mixture weight updates change R(omega) each iteration, which prevents ELBO monotonicity.



[OK] idx 127 prop 0.30  est_afr=0.686  true_afr=0.700


In [15]:
dats$true_vars_afr

[1] 546 144

In [16]:
dats$true_vars_eur

[1] 957 737

In [17]:
mix_refpanel$s

$cs
NULL

$coverage
NULL

$requested_coverage
[1] 0.95